# Daily reconstruction RMSE by period and weather lag

The two tables compare weather input windows of 1–30 days for the
weather-only and hybrid models. Each value is the daily national RMSE
of the ten-member ensemble-mean reconstruction for the indicated dates.
Both models use the same observed national deaths over 13 October 2014
to 28 December 2025.

The seasonal and calendar windows follow notebook 04, including its
combined 2023–2025 summary. Influenza season spans ISO weeks 40–20
and summer spans weeks 15–38. The weather-only model uses air
temperature and the hybrid model also uses dew point and SARI inputs.
Both tables use the same colour scale and outline the nine-day row.

Full execution evaluates 600 saved checkpoints on HPC.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys

cwd = Path.cwd()
repo = next(
    (
        path for path in (cwd, *cwd.parents)
        if (path / "configs" / "training.example.yaml").is_file()
    ),
    None,
)
if repo is None:
    raise FileNotFoundError("Open the notebook from the sari-weather-mortality checkout.")

sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load helpers from this checkout even if Jupyter cached another notebooks package.
utils_path = repo / "notebooks" / "utils.py"
spec = spec_from_file_location("sari_notebook_utils", utils_path)
notebook_utils = module_from_spec(spec)
spec.loader.exec_module(notebook_utils)

ANALYSIS_PERIOD = notebook_utils.ANALYSIS_PERIOD
daily_index = notebook_utils.daily_index
ensemble = notebook_utils.ensemble
national_daily = notebook_utils.national_daily
DEVICE = notebook_utils.DEVICE

from sari_t.inference import InferenceModel
from sari_t.run.checkpoint import load_checkpoint, load_run_config, restore_predictor

from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle

## Load models for inspection

The next cell loads the selected member and window for each setup onto
CPU. Inspect the hybrid predictor as `model` or another predictor as
`models[setup]`.

In [ ]:
inspection_member = 0
inspection_window = 9
setups = {
    "Weather only (temperature)": "temperature_kernel_weather_only",
    "Hybrid (temperature + dew point)": "temperature_kernel_hybrid",
}
windows = range(1, 31)
sweeps = {}
for label, group in setups.items():
    runs = {}
    for run in sorted((repo / "runtime" / "training" / "sets" / group).iterdir()):
        run = run.resolve()
        config = load_run_config(run / "config.yaml")
        lag = config.model_config.kernel_T_days
        if lag in runs:
            raise ValueError(f"Multiple {lag}-day runs in {group}")
        paths = [run / "checkpoints" / f"final_{member:03d}.pt" for member in range(10)]
        if not all(path.is_file() for path in paths):
            raise FileNotFoundError(f"Incomplete ensemble for {run}")
        runs[lag] = (config, paths)
    if set(runs) != set(windows):
        raise ValueError(f"{group} must contain one run for each window in {list(windows)}")
    sweeps[label] = runs

models = {
    label: restore_predictor(
        load_checkpoint(runs[inspection_window][1][inspection_member]), device="cpu",
    ).eval()
    for label, runs in sweeps.items()
}
model = models["Hybrid (temperature + dew point)"]
model

## Evaluate ensemble-mean RMSE by period

In [ ]:
dates = daily_index()
years = dates.year.to_numpy()
iso_weeks = dates.isocalendar().week.to_numpy(dtype=int)
masks = {
    "Overall": np.ones(len(dates), dtype=bool),
    "Influenza\nseason": (iso_weeks >= 40) | (iso_weeks <= 20),
    "Summer": (iso_weeks >= 15) & (iso_weeks <= 38),
    "Pre-pandemic\n2014–2019": years <= 2019,
    "Pandemic\n2020–2022": (years >= 2020) & (years <= 2022),
    "2023–2024": (years >= 2023) & (years <= 2024),
    "2025": years == 2025,
    "2023–2025": years >= 2023,
}

def period_rmse(predicted, observed):
    squared_errors = (predicted - observed).square().numpy()
    return {window: np.sqrt(squared_errors[mask].mean())
            for window, mask in masks.items()}

rmse_tables = {}
observed = None
for setup, runs in sweeps.items():
    rows = []
    for window in windows:
        config, paths = runs[window]
        print(f"{setup}, {window} input days", flush=True)
        inference_model = InferenceModel.from_checkpoint(
            paths[0], data_config=config.data_config, period=ANALYSIS_PERIOD, device=DEVICE,
        )
        reconstruction = ensemble(inference_model, paths, national_daily)
        del inference_model

        registered = reconstruction["Observed"]
        if registered.numel() != len(dates):
            raise ValueError(f"Unexpected mortality window for {setup}, {window} days")
        if observed is None:
            observed = registered
        elif not registered.equal(observed):
            raise ValueError(f"Different observed deaths for {setup}, {window} days")
        rows.append(period_rmse(reconstruction["Predicted"], observed))

    rmse_tables[setup] = pd.DataFrame(rows, index=pd.Index(windows, name="Weather input days"))

weather_only_rmse = rmse_tables["Weather only (temperature)"]
hybrid_rmse = rmse_tables["Hybrid (temperature + dew point)"]

## Weather-only and hybrid RMSE tables

In [ ]:
values = np.stack([table.to_numpy() for table in rmse_tables.values()])
if not np.isfinite(values).all():
    raise ValueError("The RMSE tables contain non-finite values.")

norm = Normalize(vmin=values.min(), vmax=values.max())
cmap = plt.get_cmap("magma_r")
reference_window = 9
with plt.rc_context({"font.family": "DejaVu Serif"}):
    for setup, table in rmse_tables.items():
        data = table.to_numpy()
        fig, ax = plt.subplots(figsize=(12, 12), layout="constrained")
        heatmap = ax.imshow(data, cmap=cmap, norm=norm, aspect="auto")
        ax.set_title(f"{setup} · Daily RMSE by period")
        ax.set_ylabel(table.index.name)
        ax.set_xticks(np.arange(len(table.columns)), table.columns)
        ax.set_yticks(np.arange(len(table.index)), table.index)
        ax.xaxis.tick_top()
        ax.tick_params(axis="both", length=0, labelsize=9)
        ax.tick_params(axis="x", pad=7)

        for (row, col), value in np.ndenumerate(data):
            red, green, blue, _ = cmap(norm(value))
            luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
            ax.text(col, row, f"{value:.1f}", ha="center", va="center",
                    color="black" if luminance > 0.5 else "white", fontsize=8)

        reference_row = table.index.get_loc(reference_window)
        ax.add_patch(Rectangle((-0.49, reference_row - 0.49), len(table.columns) - 0.02,
                               0.98, fill=False, edgecolor="#2A5C9E", linewidth=1.6))
        ax.get_yticklabels()[reference_row].set_fontweight("bold")
        fig.colorbar(heatmap, ax=ax, shrink=0.8, label="RMSE (deaths/day)")
        plt.show()